# 005 - SQL Editor for Notebook · 데모

Jupyter 셀 안에서 동작하는 single-file SQL 편집기. **좌측 entity 트리** + **우측 SQL 입력** + **컨텍스트 인식 자동완성·제안**.

## ⚡ 빠른 시작 — 다음 셀 실행

셀 output 에 SQL 편집기가 임베드됩니다. 아래 동작들을 직접 시도해보세요:

- 입력창에 `SELECT ` 까지 → 컬럼 + `*` + 함수 추천
- `SELECT * FROM ` → 테이블 추천 (users · orders · products)
- `SELECT * FROM orders WHERE ` → 컬럼 추천
- `SELECT * FROM orders WHERE orders.` → orders 컬럼만 한정
- 좌측 컬럼 더블클릭 → 커서 위치에 컬럼명 삽입
- `Ctrl+Space` → 자동완성 강제 호출
- ↑↓ 이동 · `Enter` 또는 `Tab` 확정 · `Esc` 닫기


In [5]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from sql_editor import SQLEditor

editor = SQLEditor()
editor.add_table("users", [
    ("id", "INTEGER", "PK"),
    ("name", "TEXT"),
    ("email", "TEXT"),
    ("created_at", "DATETIME"),
], description="사용자 마스터")
editor.add_table("products", [
    ("id", "INTEGER", "PK"),
    ("sku", "TEXT"),
    ("name", "TEXT"),
    ("price", "REAL"),
    ("category", "TEXT"),
], description="상품 카탈로그")
editor.add_table("orders", [
    ("id", "INTEGER", "PK"),
    ("user_id", "INTEGER", "FK → users.id"),
    ("product_id", "INTEGER", "FK → products.id"),
    ("amount", "REAL"),
    ("status", "TEXT", "pending|paid|shipped|cancelled"),
    ("ordered_at", "DATETIME"),
], description="주문 트랜잭션")

editor.set_query(
    "-- 사용자별 결제 합계 (샘플)\n"
    "SELECT u.name, SUM(o.amount) AS total\n"
    "FROM users u\n"
    "JOIN orders o ON o.user_id = u.id\n"
    "WHERE o.status = 'paid'\n"
    "GROUP BY u.name\n"
    "ORDER BY total DESC\n"
    "LIMIT 10;"
)
editor.show()


---

## 1️⃣ `from_dict` — 한 번에 여러 테이블 등록

타입 정보 없이 컬럼명만 빠르게 등록할 때.


In [6]:
ed_dict = SQLEditor()
ed_dict.from_dict({
    "customers": ["id", "name", "phone", "region"],
    "invoices":  ["id", "customer_id", "total", "due_date"],
    "payments":  ["id", "invoice_id", "paid_at", "method"],
})
ed_dict.set_query("SELECT * FROM customers WHERE region = 'KR';")
ed_dict.show()


---

## 2️⃣ `from_sqlite` — SQLite DB 자동 introspection

stdlib `sqlite3` 만 사용. 임시 DB 를 만들고 자동으로 스키마를 추출합니다.


In [7]:
import sqlite3, tempfile, os

# 임시 SQLite DB 생성
db_path = os.path.join(tempfile.gettempdir(), "sql_editor_demo.db")
conn = sqlite3.connect(db_path)
conn.executescript('''
DROP TABLE IF EXISTS departments;
DROP TABLE IF EXISTS employees;
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    location TEXT
);
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department_id INTEGER,
    hired_at TEXT,
    salary REAL,
    FOREIGN KEY (department_id) REFERENCES departments(id)
);
INSERT INTO departments VALUES
    (1, '엔지니어링', '서울'),
    (2, '디자인', '서울'),
    (3, '리스크', '부산');
INSERT INTO employees VALUES
    (1, '김개발', 1, '2024-03-01', 7000),
    (2, '이디자인', 2, '2024-05-15', 6500),
    (3, '박분석', 3, '2023-11-20', 8000);
''')
conn.commit()
conn.close()
print(f"임시 DB 생성: {db_path}")

ed_sqlite = SQLEditor().from_sqlite(db_path)
ed_sqlite.set_query(
    "-- 부서별 평균 급여\n"
    "SELECT d.name AS dept, AVG(e.salary) AS avg_salary\n"
    "FROM departments d\n"
    "JOIN employees e ON e.department_id = d.id\n"
    "GROUP BY d.name;"
)
ed_sqlite.show()


임시 DB 생성: /var/folders/nk/m8nk3pvj3vb0p_6xhdsgyj240000gn/T/sql_editor_demo.db


---

## 3️⃣ 워크플로 — 작성한 SQL 을 실제로 실행

에디터 자체는 **쿼리 실행 기능을 의도적으로 포함하지 않습니다** — 실행 인터페이스는 사용자의 DB 연결에 위임. 위 SQLite DB 와 pandas 로 결과를 확인해봅시다.


In [8]:
import pandas as pd

# 위 임시 DB 재사용
conn = sqlite3.connect(db_path)

# 에디터에서 [SQL 복사] 한 쿼리를 여기 붙여넣기 (또는 직접 작성)
sql = """
SELECT d.name AS dept,
       COUNT(e.id) AS headcount,
       AVG(e.salary) AS avg_salary,
       MAX(e.salary) AS max_salary
FROM departments d
JOIN employees e ON e.department_id = d.id
GROUP BY d.name
ORDER BY avg_salary DESC;
"""
df = pd.read_sql(sql, conn)
conn.close()
df


,dept,headcount,avg_salary,max_salary
0,리스크,1,8000.0,8000.0
1,엔지니어링,1,7000.0,7000.0
2,디자인,1,6500.0,6500.0


---

## 4️⃣ `from_dataframes` — pandas DataFrame 에서 스키마 자동 추출

분석 노트북에서 이미 가지고 있는 DataFrame 들로 SQL 을 짤 때 편함 (이 자체로 SQL 을 실행해주진 않지만, **컬럼 이름·타입을 빠르게 트리에 띄우는 용도**).


In [9]:
df_users = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "username": ["alice", "bob", "charlie", "dana"],
    "signup_date": pd.to_datetime(["2024-01-15", "2024-02-20", "2024-03-10", "2024-04-01"]),
    "is_active": [True, True, False, True],
})
df_events = pd.DataFrame({
    "event_id": range(1, 11),
    "user_id": [1, 1, 2, 3, 1, 4, 2, 4, 3, 1],
    "event_type": ["click", "view", "click", "purchase", "view",
                   "click", "view", "purchase", "view", "click"],
    "value": [None, None, None, 39000, None, None, None, 55000, None, None],
})

ed_pd = SQLEditor().from_dataframes({
    "df_users": df_users,
    "df_events": df_events,
})
ed_pd.set_query("SELECT * FROM df_events WHERE event_type = 'purchase';")
ed_pd.show()


---

## 5️⃣ `save_html` — 정적 HTML 파일로 저장 (반출용)

Jupyter 신뢰 정책으로 셀에서 스크립트가 차단되거나, 다른 사람에게 단순 공유할 때.


In [10]:
out_path = os.path.abspath("sql_editor_snapshot.html")
editor.save_html(out_path)
print(f"✓ 저장됨: {out_path}")
print(f"  크기: {os.path.getsize(out_path):,} bytes")
print("  브라우저로 직접 열면 동일한 위젯이 동작합니다 (외부 네트워크 의존 0)")


✓ 저장됨: /Users/yoon-gu/repos/sandbox-sentinel/005-sql-editor-notebook/examples/sql_editor_snapshot.html
  크기: 23,190 bytes
  브라우저로 직접 열면 동일한 위젯이 동작합니다 (외부 네트워크 의존 0)


---

## 컨텍스트 자동완성 동작 메모

| 컨텍스트 | 추천 |
|---|---|
| `FROM ...` / `JOIN ...` | 테이블 |
| `SELECT ...` | 컬럼 + `*` + 함수 |
| `WHERE ...` / `AND` / `OR` / `ON` / `HAVING` | 컬럼 |
| `GROUP BY ...` / `ORDER BY ...` | 컬럼 |
| `table_name.` | 해당 테이블 컬럼만 한정 |
| 시작/모호 | 키워드 + 함수 + 테이블 + 컬럼 종합 |

## 키 단축

- 입력 1글자 이상 → 자동 popup · `Ctrl+Space` → 강제 호출
- ↑↓ 이동 · `Enter` 또는 `Tab` 확정 · `Esc` 닫기
- popup 닫혔을 때 `Tab` → 2칸 들여쓰기 삽입
- 좌측 트리: 테이블명/컬럼명 더블클릭 → 커서 위치에 삽입
